# Lesson 04 — Color Grading with LUTs

## Why
Professional color grading is built on LUTs. A LUT maps every input value to an output value per channel — giving you complete tonal control with zero performance cost.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread('sample.jpg')
b, g, r = cv2.split(img)

def apply_lut(channel, mapping_fn):
    lut = np.array([mapping_fn(i) for i in range(256)], dtype=np.uint8)
    return cv2.LUT(channel, lut)

# Teal-Orange grade (cinema look)
# Shadows → teal (boost B in darks), Highlights → orange (boost R in brights)
def r_teal_orange(i): return int(min(255, i * 1.1 + 5)) if i > 128 else int(max(0, i * 0.9))
def b_teal_orange(i): return int(min(255, i * 1.1 + 10)) if i < 128 else int(max(0, i * 0.85))

teal_orange = cv2.merge([
    apply_lut(b, b_teal_orange),
    g,
    apply_lut(r, r_teal_orange)
])

# Faded film look: lift shadows (blacks don't go to 0)
def faded(i): return int(i * 0.85 + 20)
faded_img = cv2.merge([apply_lut(b, faded), apply_lut(g, faded), apply_lut(r, faded)])

# High contrast S-curve (darken darks, brighten brights)
def s_curve(i):
    x = i / 255.0
    y = x * x * (3 - 2*x)  # smoothstep — S-shape
    return int(y * 255)
s_img = cv2.merge([apply_lut(b, s_curve), apply_lut(g, s_curve), apply_lut(r, s_curve)])

grades = {'Original': img, 'Teal-Orange': teal_orange, 'Faded Film': faded_img, 'S-Curve': s_img}
fig, axes = plt.subplots(1, 4, figsize=(22, 6))
for ax, (name, im) in zip(axes, grades.items()):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.set_title(name, fontsize=13); ax.axis('off')
plt.suptitle('Color grading with LUTs — O(1) per pixel regardless of complexity', fontsize=12)
plt.tight_layout(); plt.show()

## Key Takeaway
LUTs are O(1) — no matter how complex your curve formula, applying it costs a single array lookup per pixel. This is why every real-time color grading tool uses them.